# Module 07: Preprocessing with fMRIPrep (Docker)

This notebook walks through running fMRIPrep — a robust, standardized fMRI preprocessing pipeline — using Docker.

**Learning objectives:**
- Understand the fMRIPrep preprocessing steps
- Run fMRIPrep for a single subject via Docker
- Understand key command-line flags
- Know what output files to expect

## 1. What Does fMRIPrep Do?

fMRIPrep is a BIDS-App that performs minimal, robust preprocessing:

| Step | Description |
|------|-------------|
| Brain extraction | Remove non-brain tissue (T1w) |
| Surface reconstruction | FreeSurfer cortical surfaces |
| Spatial normalization | Register T1w to MNI space |
| Motion correction | Realign BOLD volumes |
| Susceptibility distortion | Correct EPI distortions |
| Confound estimation | FD, DVARS, CompCor, etc. |

In [ ]:
import subprocess
import pathlib
import os

# Check Docker availability
result = subprocess.run(['docker', '--version'], capture_output=True, text=True)
if result.returncode == 0:
    print('Docker available:', result.stdout.strip())
else:
    print('Docker not found. Install from https://docs.docker.com/get-docker/')

## 2. FreeSurfer License

fMRIPrep requires a FreeSurfer license (free for research use).

1. Register at https://surfer.nmr.mgh.harvard.edu/registration.html
2. Download `license.txt`
3. Place it somewhere accessible (e.g., `~/freesurfer_license.txt`)

In [ ]:
# Check for FreeSurfer license
license_paths = [
    pathlib.Path.home() / 'freesurfer_license.txt',
    pathlib.Path('/opt/freesurfer/license.txt'),
    pathlib.Path('/usr/local/freesurfer/license.txt'),
]
for p in license_paths:
    if p.exists():
        print(f'Found license at: {p}')
        break
else:
    print('License not found. Download from: https://surfer.nmr.mgh.harvard.edu/registration.html')

## 3. The fMRIPrep Docker Command

The basic command structure:
```bash
docker run --rm \
  -v /path/to/bids:/data:ro \
  -v /path/to/output:/out \
  -v /path/to/license.txt:/opt/freesurfer/license.txt:ro \
  nipreps/fmriprep:23.1.4 \
  /data /out participant \
  --participant-label sub-01 \
  --output-spaces MNI152NLin2009cAsym:res-2 T1w \
  --dummy-scans 4 \
  --fd-spike-threshold 0.5 \
  --skip-bids-validation
```

In [ ]:
# Build the command programmatically
bids_dir = pathlib.Path('../data/example_bids').resolve()
output_dir = pathlib.Path('/tmp/fmriprep_output').resolve()
license_file = pathlib.Path.home() / 'freesurfer_license.txt'

cmd = [
    'docker', 'run', '--rm',
    '-v', f'{bids_dir}:/data:ro',
    '-v', f'{output_dir}:/out',
    '-v', f'{license_file}:/opt/freesurfer/license.txt:ro',
    '--memory=8g', '--cpus=4',
    'nipreps/fmriprep:23.1.4',
    '/data', '/out', 'participant',
    '--participant-label', '01',
    '--output-spaces', 'MNI152NLin2009cAsym:res-2', 'T1w',
    '--dummy-scans', '4',
    '--fd-spike-threshold', '0.5',
    '--skip-bids-validation',
    '--nthreads', '4',
]
print('Command to run:')
print(' \\\n  '.join(str(x) for x in cmd))

## 4. Key fMRIPrep Flags

| Flag | Description |
|------|-------------|
| `--output-spaces` | Output spaces for BOLD (MNI, T1w, fsaverage) |
| `--dummy-scans N` | Remove first N volumes (magnetization equilibration) |
| `--fd-spike-threshold` | FD threshold for spike detection (mm) |
| `--skip-bids-validation` | Skip BIDS validator (useful for synthetic data) |
| `--use-aroma` | Run ICA-AROMA for noise removal |
| `--return-all-components` | Return all ICA components |
| `--nthreads` | Number of CPU threads |
| `--mem-mb` | Memory limit (MB) |

## 5. Expected Output Structure

After fMRIPrep completes:
```
fmriprep_output/
├── fmriprep/
│   ├── sub-01/
│   │   ├── anat/
│   │   │   ├── sub-01_desc-preproc_T1w.nii.gz
│   │   │   ├── sub-01_space-MNI152NLin2009cAsym_desc-preproc_T1w.nii.gz
│   │   │   └── sub-01_desc-brain_mask.nii.gz
│   │   └── func/
│   │       ├── sub-01_task-emotionreg_run-01_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz
│   │       ├── sub-01_task-emotionreg_run-01_desc-confounds_timeseries.tsv
│   │       └── sub-01_task-emotionreg_run-01_space-MNI152NLin2009cAsym_boldref.nii.gz
│   └── sub-01.html  ← Visual QC report
└── freesurfer/
    └── sub-01/      ← FreeSurfer outputs
```

## 6. Common Errors and Solutions

| Error | Cause | Solution |
|-------|-------|----------|
| `license.txt not found` | Missing FreeSurfer license | Provide correct path to license |
| `Out of memory` | Dataset too large | Increase `--mem-mb` or reduce `--nthreads` |
| `Cannot connect to Docker daemon` | Docker not running | Start Docker Desktop |
| `BIDS validation failed` | Dataset not BIDS compliant | Fix BIDS issues or use `--skip-bids-validation` |
| `Fieldmap not found` | Missing fmap data | Provide fieldmaps or use `--ignore fieldmaps` |

## Summary

In this module you learned:
- The fMRIPrep preprocessing pipeline steps
- How to run fMRIPrep via Docker
- Key command-line flags and their effects
- Expected output structure

**Next:** Module 08 covers building custom Nipype workflows, and Module 09 covers exploring fMRIPrep outputs.